## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,15186496707,4194773,2026-06-24,2026-06-24,1,NI,5685.500000000,BLEND3_3,4194773,"{""pessoas"":[{""nome"":""DIEGO GOMES DE OLIVEIRA"",...",...,32,1778785248777,"{""imobiliaria_id"":""37880"",""imovel"":{""tipo"":""RE...",2026-06-24 18:00:46+00:00,1782324042304564618,15186496707,B,2026-06-24 18:25:11.462000+00:00,2026-06-24 12:11:49+00:00,APROVAR
1,14571693630,4211411,2026-07-01,2026-07-01,1,NI,3014.000000000,BLEND3_3,4211411,"{""pessoas"":[{""nome"":""BRUNA CRISTINE LUCIANO RO...",...,32,1778785248777,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",2026-07-02 08:01:27+00:00,1782979285685779309,14571693630,B,2026-07-02 08:11:58.239000+00:00,2026-07-01 18:34:19+00:00,DERIVAR
2,4930701090,4225169,2026-07-07,2026-07-07,1,E,1164.500000000,BLEND3_3,4225169,"{""pessoas"":[{""nome"":""JOAO PEDRO DALBAO XAVIER""...",...,34,1778785248777,"{""valor_aluguel"":800,""matchmaking_on"":false,""p...",2026-07-08 08:00:27+00:00,1783497623436579966,04930701090,C,2026-07-08 08:01:59.857000+00:00,2026-07-07 15:57:58+00:00,APROVAR
3,7868479990,4229237,2026-06-24,2026-06-24,1,NI,0E-9,BLEND3_3,4229237,"{""pessoas"":[{""nome"":"""",""documento"":""0786847999...",...,33,1778785248777,"{""valor_aluguel"":""1385"",""imobiliaria_id"":17461...",2026-06-25 08:00:36+00:00,1782374434171539453,07868479990,E,2026-06-25 08:00:46.641000+00:00,2026-06-24 17:28:14+00:00,REPROVAR
4,42453374869,4236208,2026-06-19,2026-06-19,1,F,1301.500000000,BLEND3_3,4236208,"{""pessoas"":[{""nome"":""ANE EMANUELLE GANDARA SOA...",...,34,1778785248777,"{""valor_aluguel"":""650"",""imobiliaria_id"":22913,...",2026-06-19 18:00:49+00:00,1781892045796271857,42453374869,C,2026-06-19 18:08:45.621000+00:00,2026-06-19 11:23:30+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112286,14382376777,4373009,2026-07-14,2026-07-14,1,NI,2329.000000000,BLEND3_3,4373009,"{""pessoas"":[{""nome"":""ANDRESSA CRISTINA DOS SAN...",...,33,1778785248777,"{""valor_aluguel"":1550,""matchmaking_on"":false,""...",2026-07-15 08:00:37+00:00,1784102435134541212,14382376777,E,2026-07-15 08:04:36.870000+00:00,2026-07-14 17:31:35+00:00,APROVAR
112287,50126922845,4373053,2026-07-14,2026-07-14,1,D,1301.500000000,BLEND_4,4373053,"{""pessoas"":[{""nome"":""BEATRIZ FERREIRA RUIZ MAR...",...,34,1778785248777,"{""principalDocument"":""50126922845"",""imobiliari...",2026-07-15 08:00:37+00:00,1784102435228792327,50126922845,C,2026-07-15 08:04:36.885000+00:00,2026-07-14 17:37:19+00:00,APROVAR
112288,59948786858,4373334,2026-07-14,2026-07-14,1,NI,2466.000000000,BLEND3_3,4373334,"{""pessoas"":[{""nome"":""DIEGO DE QUEIROZ ALEXANDR...",...,33,1778785248777,"{""valor_aluguel"":1900,""matchmaking_on"":false,""...",2026-07-15 08:00:37+00:00,1784102435739324333,59948786858,E,2026-07-15 08:04:36.966000+00:00,2026-07-14 18:25:56+00:00,REPROVAR
112289,48436155874,4373427,2026-07-14,2026-07-14,1,NI,1301.500000000,BLEND3_3,4373427,"{""pessoas"":[{""nome"":""RAISSA VITORIA MARES PERE...",...,32,1778785248777,"{""valor_aluguel"":650,""matchmaking_on"":false,""p...",2026-07-15 08:00:37+00:00,1784102435910844413,48436155874,B,2026-07-15 08:04:36.993000+00:00,2026-07-14 19:24:15+00:00,APROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                95153
BLEND_REGRESSAO_2026     5555
BLEND_4                  5043
HVA3                     4403
BVS_CUSTOM               1283
HVA4                      694
BVS_CUSTOM_V2             125
HFT1                       19
                           16
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    107949
2      3981
3       322
4        34
5         3
8         1
6         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961333
2    0.035453
3    0.002868
4    0.000303
5    0.000027
8    0.000009
6    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [13]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [14]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [15]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [16]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
685,0.0,0.172243
686,0.0,0.151584
1779,0.0,0.104675
1781,0.0,0.130864
2922,NaN,0.093633
...,...,...
108524,0.0,0.130864
108525,0.0,0.000000
109678,NaN,NaN
110765,NaN,0.123300


## Escoragem Blend4

In [17]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [18]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [19]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [20]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] <= 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,15186496707,4194773,2026-06-24,2026-06-24,1,NI,BLEND3_3,5996940,32,15186496707,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,5685.5
1,14571693630,4211411,2026-07-01,2026-07-01,1,NI,BLEND3_3,6033302,32,14571693630,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,3014.0
2,4930701090,4225169,2026-07-07,2026-07-07,1,E,BLEND3_3,6061282,34,04930701090,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1164.5
3,7868479990,4229237,2026-06-24,2026-06-24,1,NI,BLEND3_3,6000135,33,07868479990,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
4,42453374869,4236208,2026-06-19,2026-06-19,1,F,BLEND3_3,5973930,34,42453374869,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1301.5


In [21]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111759
E_BVS        532
dtype: int64

In [22]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [23]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [24]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [25]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [26]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,15186496707,4194773,2026-06-24,2026-06-24,1,NI,BLEND3_3,5996940,32,15186496707,...,0.000000,0.05,0.0,-0.522338,0.000000,0.000000,0.478561,521.0,A,A
1,14571693630,4211411,2026-07-01,2026-07-01,1,NI,BLEND3_3,6033302,32,14571693630,...,0.000000,-0.35,1.0,-0.070249,0.000000,0.000000,0.496849,503.0,A,A
2,4930701090,4225169,2026-07-07,2026-07-07,1,E,BLEND3_3,6061282,34,04930701090,...,0.000000,-0.80,0.0,0.567787,-0.020067,-0.614615,0.538058,462.0,B,C
3,7868479990,4229237,2026-06-24,2026-06-24,1,NI,BLEND3_3,6000135,33,07868479990,...,0.000000,0.00,0.0,0.000000,0.000000,-0.081865,0.508237,492.0,A,None
4,42453374869,4236208,2026-06-19,2026-06-19,1,F,BLEND3_3,5973930,34,42453374869,...,0.000000,-0.80,0.0,0.153472,0.000000,0.320622,0.526529,473.0,B,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112286,14382376777,4373009,2026-07-14,2026-07-14,1,NI,BLEND3_3,6096786,33,14382376777,...,0.000000,0.05,2.0,0.520365,0.000000,0.768522,0.564676,435.0,C,E
112287,50126922845,4373053,2026-07-14,2026-07-14,1,D,BLEND_4,6096847,34,50126922845,...,-0.361516,-0.80,0.0,1.086928,0.000000,0.161927,0.574503,425.0,C,C
112288,59948786858,4373334,2026-07-14,2026-07-14,1,NI,BLEND3_3,6097191,33,59948786858,...,0.000000,-0.75,3.0,0.752204,0.000000,0.892747,0.582138,418.0,C,E
112289,48436155874,4373427,2026-07-14,2026-07-14,1,NI,BLEND3_3,6097302,32,48436155874,...,0.000000,-0.75,0.0,0.153472,0.000000,-0.198630,0.508312,492.0,A,A


In [27]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           16
BLEND3_3                95153
BLEND_4                  5043
BLEND_REGRESSAO_2026     5555
BVS_CUSTOM               1283
BVS_CUSTOM_V2             125
HFT1                       19
HVA3                     4403
HVA4                      694
dtype: int64

In [28]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [29]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111759
E_BVS        532
dtype: int64

In [30]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           16
BLEND3_3                95153
BLEND_4                  5043
BLEND_REGRESSAO_2026     5555
BVS_CUSTOM               1283
BVS_CUSTOM_V2             125
HFT1                       19
HVA3                     4403
HVA4                      694
dtype: int64

In [31]:
cp_final_df.groupby("model", dropna=False).size()

model
                           16
BLEND3_3                95153
BLEND_4                  5043
BLEND_REGRESSAO_2026     5555
BVS_CUSTOM               1283
BVS_CUSTOM_V2             125
HFT1                       19
HVA3                     4403
HVA4                      694
dtype: int64

In [32]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111759
E_BVS        532
dtype: int64

In [33]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)